# Coupling energy system models with life-cycle assessment

This notebook aims to show how to generate Life-Cycle Assessment (LCA) impact scores to be used in any technology-rich (bottom-up) Energy System Model (ESM). LCA impact scores can be used within modeling constraints, e.g., upper limit on life-cycle greenhouse gas emissions, or in the objective function, e.g., minimizing the total damage on human health. LCA impact scores are generated and integrated in your ESM using the Python package [_mescal_](https://mescal.readthedocs.io/en/latest/). The latter applies several transformations on LCA data to ensure the alignement between your ESM and LCA data. For instance, _mescal_ performs double-counting removal (to avoid the overestimation of flows that are already represented in your ESM, e.g., energy flows), technological parameters harmonization (e.g., lifetime, efficiency, capacity factors), and normalization (to ease the solving process).

In this notebook, we illustrate the use of _mescal_ with a toy ESM. We might employ a prospective LCA database from [_premise_](https://premise.readthedocs.io/en/latest/introduction.html), but the overall methodology is agnostic to the used LCA database and to the ESM you are using. You should simply adapt the input data files accordingly.

In this notebook, we show how to:
- Create the ESM database (i.e., the database containing the datasets corresponding to the technologies and resources of the ESM) in your brightway2 project
- Perform life-cycle impact assessment and contribution analyses
- (Optional) Create the .mod and .dat files for your ESM (AMPL)
- Visualize the results

## Project setup

To be able to run the following script, you need to have already set up a Brightway2 project with ecoinvent (and optionally premise) databases. You can refer to this [documentation](https://learn.brightway.dev/en/latest/content/chapters/BW2/BW2_introduction.html) to set up Brightway2 and this [notebook](https://github.com/polca/premise/blob/master/examples/examples.ipynb) to set up premise databases in your Brightway2 project.

In [ ]:
# Import the required libraries
from mescal import *
import pandas as pd
import bw2data as bd

In [ ]:
##### YOUR SETTINGS #####
location = 'DK'  # Set the ecoinvent location code corresponding to your ESM here (e.g., 'DK' for Denmark)
year = 2050  # Set the year here (e.g., 2050)
brightway_project_name = 'brightcon-2026'  # Set the name of your brightway project
lca_db_name = f"ecoinvent_cutoff_3.12_image_SSP2-M_{year}"  # Set the name of your main LCI database (e.g., ecoinvent or premise database)
esm_db_name = f'toy_{location}_{year}'  # Set the name of the new database with the ESM datasets
lcia_methods=['EF v3.1']  # Set the list of LCIA methods for which you want indicators (they must be registered in your brightway project)

**Note**: you can download IMPACT World+ methods in your brightway project following this [notebook](https://github.com/matthieu-str/mescal/blob/master/dev/download_impact_world_plus.ipynb). Regionalized versions of ecoinvent and IMPACT World+ methods can be downloaded from this [repository](https://github.com/matthieu-str/Regiopremise).

In [ ]:
# Set up your Brightway project
bd.projects.set_current(brightway_project_name)

In [ ]:
# Load the LCI database from your brightway project
lca_db = Database(lca_db_name, create_pickle=True)

In [ ]:
# If necessary, add missing CPC categories to the database
lca_db.add_CPC_categories(overwrite_existing_CPC=True)

## Input data

In [ ]:
path_to_input_files = './input_data/'

Complete input data files can be retrieved from [mescal documentation](https://mescal.readthedocs.io/en/latest/content/user_inputs.html)

In [ ]:
mapping = pd.read_csv(path_to_input_files+'mapping.csv')
unit_conversion = pd.read_excel(path_to_input_files+'unit_conversion_factors.xlsx')
model = pd.read_csv(path_to_input_files+'model.csv')
technology_compositions = pd.read_csv(path_to_input_files+'technology_compositions.csv')
lifetime = pd.read_csv(path_to_input_files+'lifetime.csv')
efficiency = pd.read_csv(path_to_input_files+'efficiency.csv')
mapping_esm_flows_to_cpc = pd.read_csv(path_to_input_files+'mapping_esm_flows_to_CPC.csv')
impact_abbrev = pd.read_csv(path_to_input_files+'impact_abbrev.csv')

In [ ]:
mapping.Database = lca_db_name  # set the chosen database name in the mapping file

## Initialize the ESM class

In [ ]:
esm = ESM(
    mapping=mapping,
    unit_conversion=unit_conversion,
    model=model,
    main_database=lca_db,
    esm_db_name=esm_db_name,
    esm_location=location,
    technology_compositions=technology_compositions,
    lifetime=lifetime,
    efficiency=efficiency,
    mapping_esm_flows_to_CPC_cat=mapping_esm_flows_to_cpc,

    regionalize_foregrounds=['Operation', 'Resource'],  # types of LCI datasets that will be regionalized
    locations_ranking=['DK', 'RER', 'WEU', 'CEU', 'GLO', 'RoW'],  # order of preference for locations when regionalizing
    results_path_file='./output_data/',
)

In [ ]:
# Update mapping dataframe with better locations
esm.change_location_mapping_file()

In [ ]:
esm.main_database.test_mapping_file(esm.mapping)  # test the mapping file

In [ ]:
esm.clean_inputs()

In [ ]:
esm.check_inputs()

## Create the ESM database in your Brightway2 project

In [ ]:
esm.create_esm_database()

## Compute LCA impact scores and perform contribution analysis

In [ ]:
impact_scores, contrib_analysis_res, _ = esm.compute_impact_scores(
    methods=lcia_methods,
    contribution_analysis='both',
)

In [ ]:
impact_scores.to_csv(esm.results_path_file+'impact_scores.csv', index=False)
contrib_analysis_res.to_csv(esm.results_path_file+'contribution_analysis.csv', index=False)

In [ ]:
impact_scores_direct_emissions, _, _ = esm.compute_impact_scores(
    methods=lcia_methods,
    assessment_type='direct emissions',  # specific metrics for direct emissions during operation
)

In [ ]:
impact_scores_direct_emissions.to_csv(esm.results_path_file+'impact_scores_direct_emissions.csv', index=False)

## Create .dat and .mod files for AMPL integration

By default, the resulting .dat and .mod files are saved in the result directory specified in the ESM class initialization (here './output_data/').

We can select a few impact categories that we want to integrate in the model. Here, we select only two categories: TTHH (Total human health) and TTEQ (Total ecosystem quality). The abbreviations are defined in the impact_abbrev.csv file.

In [ ]:
esm.normalize_lca_metrics(
    R=impact_scores,
    mip_gap=1e-6,
    lcia_methods=lcia_methods,
    impact_abbrev=impact_abbrev,
    file_name='techs_lca',
)

In [ ]:
# specific for direct emissions metrics
esm.normalize_lca_metrics(
    R=impact_scores,
    R_direct=impact_scores_direct_emissions,  # thus we specify this
    mip_gap=1e-6,
    lcia_methods=lcia_methods,
    assessment_type='direct emissions',  # and this
    impact_abbrev=impact_abbrev,
    file_name='techs_lca_direct',
)

In [ ]:
esm.generate_mod_file_ampl(
    lcia_methods=lcia_methods,
    impact_abbrev=impact_abbrev,
    file_name='objectives_lca',
)

In [ ]:
# specific for direct emissions metrics
esm.generate_mod_file_ampl(
    lcia_methods=lcia_methods,
    assessment_type='direct emissions',  # specify this
    impact_abbrev=impact_abbrev,
    file_name='objectives_lca_direct',
)

## Visualize the LCA impact scores

In [ ]:
plot = Plot(
    df_impact_scores=impact_scores,
    lifetime=lifetime,  # used to visualize infrastructure impacts per kW.year
)

In [ ]:
plot.plot_indicators_of_technologies_for_one_impact_category(
    technologies_list=[
        'PV_ROOF',
        'WIND_ONSHORE',
        'WIND_OFFSHORE',
        'CCGT',
        'CCGT_CC',
        'COAL_IGCC',
        'COAL_IGCC_CC',
        'NUCLEAR',
    ],
    impact_category=(
        'EF v3.1',
        'climate change',
        'global warming potential (GWP100)'
    ),
    metadata={
        'operation_unit': 'kWh',
        'construction_unit': 'kW',
    },
)

In [ ]:
plot.plot_indicators_of_technologies_for_several_impact_categories(
    technologies_list=[
        'PV_ROOF',
        'WIND_ONSHORE',
        'WIND_OFFSHORE',
        'CCGT',
        'CCGT_CC',
        'COAL_IGCC',
        'COAL_IGCC_CC',
        'NUCLEAR',
    ],
    impact_categories_list=[
        ('EF v3.1', 'climate change', 'global warming potential (GWP100)'),
        ('EF v3.1', 'ecotoxicity: freshwater', 'comparative toxic unit for ecosystems (CTUe)'),
        ('EF v3.1', 'land use', 'soil quality index'),
        ('EF v3.1', 'water use', 'user deprivation potential (deprivation-weighted water consumption)'),
    ],
)

In [ ]:
plot.plot_indicators_of_resources_for_several_impact_categories(
    resources_list=['COAL', 'NG_EHP', 'URANIUM'],
    impact_categories_list=[
        ('EF v3.1', 'climate change', 'global warming potential (GWP100)'),
        ('EF v3.1', 'ecotoxicity: freshwater', 'comparative toxic unit for ecosystems (CTUe)'),
        ('EF v3.1', 'land use', 'soil quality index'),
        ('EF v3.1', 'water use', 'user deprivation potential (deprivation-weighted water consumption)'),
    ],
)